# Prepare External Kidney Visium Cache

这份 notebook 会把 data/other_kidney/ 下的外部 Visium 肾癌切片转换成当前 kidney CellFM super-resolution 推理可直接复用的 cache 格式。

关键处理策略：

- 读取 filtered_feature_bc_matrix.h5
- 对重复基因名先做合并求和
- 保留全部唯一基因，不在这一步裁剪到 checkpoint 基因
- 使用 tissue_hires_image.png
- 由于原始 hires 图只有 2000px 左右，而现有 kidney 模型训练时使用的 cache 尺度接近 16000px，这里会先把 hires 图上采样到 max_dim=16000
- 然后提取 embeddings-hist.pickle

输出目录：

- data/other_kidney_cache/<slide_id>/


In [1]:
import json
import pickle
import shutil
import sys
from collections import Counter
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
from PIL import Image
from scipy import sparse

ROOT = Path(r"E:\Morpho-FM")
sys.path.append(str(ROOT / "src"))

from st_pipeline.superres.feature_extract import extract_embeddings

try:
    from IPython.display import display
except ImportError:
    display = None

Image.MAX_IMAGE_PIXELS = None

SOURCE_ROOT = ROOT / "data" / "other_kidney"
CACHE_ROOT = ROOT / "data" / "other_kidney_cache"
MODEL_DIR = ROOT / "checkpoints" / "hipt_backbone"

SLIDE_DIRS = [
    SOURCE_ROOT / "GSM6415706_GLMF1",
    SOURCE_ROOT / "GSM6415706_GLMF2",
]

TARGET_MAX_DIM = 16000
FORCE_REBUILD = False

print("SOURCE_ROOT:", SOURCE_ROOT)
print("CACHE_ROOT:", CACHE_ROOT)
print("TARGET_MAX_DIM:", TARGET_MAX_DIM)


SOURCE_ROOT: E:\Morpho-FM\data\other_kidney
CACHE_ROOT: E:\Morpho-FM\data\other_kidney_cache
TARGET_MAX_DIM: 16000


In [2]:
def display_df(df):
    if display is not None:
        display(df)
    else:
        print(df)


def read_visium_h5(h5_path: Path):
    with h5py.File(h5_path, "r") as f:
        m = f["matrix"]
        data = m["data"][:]
        indices = m["indices"][:]
        indptr = m["indptr"][:]
        shape = tuple(m["shape"][:].tolist())
        genes = [x.decode("utf-8") if isinstance(x, bytes) else str(x) for x in m["features"]["name"][:]]
        barcodes = [x.decode("utf-8") if isinstance(x, bytes) else str(x) for x in m["barcodes"][:]]
    mat = sparse.csc_matrix((data, indices, indptr), shape=shape)
    return mat, genes, barcodes


def deduplicate_gene_rows(mat: sparse.csc_matrix, genes: list[str]):
    counter = Counter(genes)
    duplicate_genes = [g for g, n in counter.items() if n > 1]
    ordered_unique = []
    seen = set()
    for g in genes:
        if g not in seen:
            ordered_unique.append(g)
            seen.add(g)

    gene_to_indices = {}
    for i, g in enumerate(genes):
        gene_to_indices.setdefault(g, []).append(i)

    rows = []
    for g in ordered_unique:
        idxs = gene_to_indices[g]
        if len(idxs) == 1:
            vec = mat[idxs[0], :].toarray()
        else:
            vec = mat[idxs, :].sum(axis=0)
        rows.append(np.asarray(vec).ravel())

    dedup = np.stack(rows, axis=0).astype(np.float32, copy=False)
    return dedup, ordered_unique, duplicate_genes


def load_spatial_positions(spatial_dir: Path):
    pos = pd.read_csv(
        spatial_dir / "tissue_positions_list.csv",
        header=None,
        names=["barcode", "in_tissue", "array_row", "array_col", "pxl_row", "pxl_col"],
    )
    sf = json.loads((spatial_dir / "scalefactors_json.json").read_text(encoding="utf-8"))
    return pos, sf


def upscale_hires_image(src_img: Path, out_img: Path, target_max_dim: int):
    img = Image.open(src_img).convert("RGB")
    w, h = img.size
    scale = float(target_max_dim) / float(max(w, h))
    if scale < 1.0:
        scale = 1.0
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))
    if (new_w, new_h) != img.size:
        img = img.resize((new_w, new_h), resample=Image.Resampling.BICUBIC)
    out_img.parent.mkdir(parents=True, exist_ok=True)
    img.save(out_img, quality=95)
    return img.size, scale


def build_slide_cache(slide_dir: Path, cache_root: Path, target_max_dim: int, force_rebuild: bool = False):
    slide_id = slide_dir.name
    out_dir = cache_root / slide_id
    out_dir.mkdir(parents=True, exist_ok=True)

    h5_files = sorted(slide_dir.glob("*filtered_feature_bc_matrix.h5"))
    if not h5_files:
        raise FileNotFoundError(f"No filtered_feature_bc_matrix.h5 found in {slide_dir}")
    h5_path = h5_files[0]
    spatial_dir = slide_dir / "spatial"
    hires_img = spatial_dir / "tissue_hires_image.png"
    if not hires_img.exists():
        raise FileNotFoundError(f"Missing hires image: {hires_img}")

    cnts_path = out_dir / "counts.npy"
    genes_path = out_dir / "genes.txt"
    spot_ids_path = out_dir / "spot_ids.txt"
    locs_path = out_dir / "locs.npy"
    radius_path = out_dir / "radius.txt"
    image_path = out_dir / "he.jpg"
    emb_path = out_dir / "embeddings-hist.pickle"
    meta_path = out_dir / "cache_meta.json"

    if force_rebuild or not (cnts_path.exists() and genes_path.exists() and spot_ids_path.exists() and locs_path.exists() and radius_path.exists()):
        mat, genes, barcodes = read_visium_h5(h5_path)
        dedup_gene_spot, unique_genes, duplicate_genes = deduplicate_gene_rows(mat, genes)
        counts = dedup_gene_spot.T

        pos, sf = load_spatial_positions(spatial_dir)
        pos = pos.set_index("barcode").loc[barcodes].copy()
        in_tissue = pos["in_tissue"].astype(int).to_numpy() == 1
        kept_barcodes = [barcodes[i] for i, ok in enumerate(in_tissue) if ok]
        counts = counts[in_tissue].astype(np.float32, copy=False)

        _, scale_up = upscale_hires_image(hires_img, image_path, target_max_dim)

        hires_scale = float(sf["tissue_hires_scalef"])
        y = pos.loc[kept_barcodes, "pxl_row"].astype(float).to_numpy() * hires_scale * scale_up
        x = pos.loc[kept_barcodes, "pxl_col"].astype(float).to_numpy() * hires_scale * scale_up
        locs = np.stack([y, x], axis=-1).astype(np.float32, copy=False)

        radius_px = float(sf["spot_diameter_fullres"]) * hires_scale * scale_up / 2.0

        np.save(cnts_path, counts)
        np.save(locs_path, locs)
        genes_path.write_text("\n".join(unique_genes), encoding="utf-8")
        spot_ids_path.write_text("\n".join(kept_barcodes), encoding="utf-8")
        radius_path.write_text(str(radius_px), encoding="utf-8")

        meta = {
            "slide_id": slide_id,
            "source_h5": str(h5_path),
            "source_hires_image": str(hires_img),
            "target_max_dim": int(target_max_dim),
            "scale_up_from_hires": float(scale_up),
            "n_genes_raw": int(len(genes)),
            "n_genes_unique": int(len(unique_genes)),
            "duplicate_genes": duplicate_genes,
            "n_spots_total": int(len(barcodes)),
            "n_spots_in_tissue": int(len(kept_barcodes)),
            "radius_px_scaled": float(radius_px),
        }
        meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    else:
        meta = json.loads(meta_path.read_text(encoding="utf-8"))

    if force_rebuild or not emb_path.exists():
        extract_embeddings(
            he_path=image_path,
            out_path=emb_path,
            device="cuda" if __import__("torch").cuda.is_available() else "cpu",
            model_dir=MODEL_DIR,
            smoothen=True,
            use_shift=False,
            force=True,
        )

    counts = np.load(cnts_path, mmap_mode="r")
    locs = np.load(locs_path, mmap_mode="r")
    radius = float(radius_path.read_text(encoding="utf-8").strip())

    mask_size_est = int(np.count_nonzero(
        (lambda r: (np.stack(np.meshgrid(
            np.arange(-int(np.ceil(r)), int(np.ceil(r)) + 1),
            np.arange(-int(np.ceil(r)), int(np.ceil(r)) + 1),
            indexing="ij"), axis=-1) ** 2).sum(-1) <= r ** 2)(radius / 16.0)
    ))

    return {
        "slide_id": slide_id,
        "cache_dir": str(out_dir),
        "counts_shape": tuple(int(x) for x in counts.shape),
        "locs_shape": tuple(int(x) for x in locs.shape),
        "radius_px": float(radius),
        "mask_size_est": int(mask_size_est),
        "scale_up_from_hires": float(meta["scale_up_from_hires"]),
        "n_genes_unique": int(meta["n_genes_unique"]),
        "n_duplicate_genes": int(len(meta["duplicate_genes"])),
    }


In [3]:
summary_rows = []
for slide_dir in SLIDE_DIRS:
    print("\nPreparing:", slide_dir.name)
    row = build_slide_cache(
        slide_dir=slide_dir,
        cache_root=CACHE_ROOT,
        target_max_dim=TARGET_MAX_DIM,
        force_rebuild=FORCE_REBUILD,
    )
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
display_df(summary_df)



Preparing: GSM6415706_GLMF1

Preparing: GSM6415706_GLMF2


,slide_id,cache_dir,counts_shape,locs_shape,radius_px,mask_size_est,scale_up_from_hires,n_genes_unique,n_duplicate_genes
0,GSM6415706_GLMF1,E:\Morpho-FM\data\other_kidney_cache\GSM641570...,"(3199, 17941)","(3199, 2)",55.836288,37,8.0,17941,2
1,GSM6415706_GLMF2,E:\Morpho-FM\data\other_kidney_cache\GSM641570...,"(3880, 17941)","(3880, 2)",55.878102,37,8.0,17941,2
